In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import h5py, os, tqdm, glob, scipy
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.49'

import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
from flax import nnx
import optax

import jaxpm
from jaxpm import hpm, camels, training, plotting, diagnostics, objectives
from jaxpm.painting import cic_paint, cic_read, compensate_cic
from jaxpm.splines import NeuralSplineFourierFilterNNX
from jaxpm.nn import MLP, ConditionedCNN
from jaxpm.objectives import ParticleLoss, FieldLoss

print(jax.default_backend())

/global/common/software/des/athomsen/flatiron/lib/python3.11/site-packages/jax_cosmo/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound


gpu


# configuration

In [3]:
parts_per_dim = 128
mesh_per_dim = parts_per_dim
mesh_shape = [mesh_per_dim] * 3

# with_latent = True
with_latent = False

# CAMELS

In [4]:
CODE = "SIMBA"
# CODE = "ASTRID"
# CODE = "IllustrisTNG"

train_dict = camels.load_CV_snapshots(
    "CV_0",
    mesh_per_dim,
    parts_per_dim,
    i_snapshots=None,
    CAMELS="/pscratch/sd/a/athomsen/flatiron/CAMELS",
    CODE=CODE,
)

vali_dict = camels.load_CV_snapshots(
    "CV_1",
    mesh_per_dim,
    parts_per_dim,
    i_snapshots=None,
    CAMELS="/pscratch/sd/a/athomsen/flatiron/CAMELS",
    CODE=CODE,
)

Loaded /pscratch/sd/a/athomsen/flatiron/CAMELS/h5/SIMBA/CV/CV_0/parts=128,mesh=128.h5
Loaded /pscratch/sd/a/athomsen/flatiron/CAMELS/h5/SIMBA/CV/CV_1/parts=128,mesh=128.h5


In [5]:
# for camels_dict in [train_dict, vali_dict]:
#     diagnostics.run_simulations(
#         camels_dict,
#         mesh_per_dim,
#         dt0=dt0,
#         i_plot=i_plot,
#         plot_dm=True,
#         plot_gas=False,
#     )

In [6]:
cosmo = train_dict["cosmo"]
solve_ode = training.get_ode_solver(mesh_per_dim, cosmo)
train_step = training.get_train_step(mesh_per_dim, cosmo)

# loss

### CAMELS ground truth

In [7]:
# general
cosmo = train_dict["cosmo"]
all_scales = train_dict["scales"]

In [8]:
i0 = 0
i1 = len(all_scales) - 1
i_ref = np.linspace(i0, i1, 5, dtype=int)[1:]

scales = all_scales[i_ref]

vcic_paint = jax.vmap(cic_paint, in_axes=(None,0,None))
vcic_read = jax.vmap(cic_read, in_axes=(0,0))

# particles
dm_poss = train_dict["dm_poss"][i_ref]
dm_vels = train_dict["dm_vels"][i_ref]

gas_poss = train_dict["gas_poss"][i_ref]
gas_vels = train_dict["gas_vels"][i_ref]

# fields
dm_mass = cosmo.Omega_c / (cosmo.Omega_b + cosmo.Omega_c)
gas_mass = 1 - dm_mass

rhos_dm = vcic_paint(jnp.zeros(mesh_shape), dm_poss, dm_mass)
deltas_dm = rhos_dm/rhos_dm.mean() - 1

rhos_gas = vcic_paint(jnp.zeros(mesh_shape), gas_poss, gas_mass)
deltas_gas = rhos_gas/rhos_gas.mean() - 1

# power spectrum
_, cls_dm = objectives.vpower_spectrum(deltas_dm)
_, cls_gas = objectives.vpower_spectrum(deltas_gas)

In [9]:
scales

array([0.30267933, 0.44367839, 0.65035996, 1.        ])

# gravity correction

In [8]:
# # general
# cosmo = train_dict["cosmo"]
# scales = train_dict["scales"]

# # particles
# dm_poss = train_dict["dm_poss"]
# dm_vels = train_dict["dm_vels"]

# gas_poss = train_dict["gas_poss"]
# gas_vels = train_dict["gas_vels"]

# # fields
# dm_mass = cosmo.Omega_c / (cosmo.Omega_b + cosmo.Omega_c)
# gas_mass = 1 - dm_mass

# # rhos_dm = vcic_paint(jnp.zeros(mesh_shape), dm_poss, dm_mass)
# rhos_dm = []
# for i in range(len(scales)):
#     rhos_dm.append(cic_paint(jnp.zeros(mesh_shape), dm_poss[i], dm_mass))
# rhos_dm = jnp.stack(rhos_dm, axis=0)

# deltas_dm = rhos_dm/rhos_dm.mean() - 1

# # rhos_gas = vcic_paint(jnp.zeros(mesh_shape), gas_poss, gas_mass)
# rhos_gas = []
# for i in range(len(scales)):
#     rhos_gas.append(cic_paint(jnp.zeros(mesh_shape), gas_poss[i], gas_mass))
# rhos_gas = jnp.stack(rhos_gas, axis=0)

# deltas_gas = rhos_gas/rhos_gas.mean() - 1

# # power spectrum
# _, cls_dm = objectives.vpower_spectrum(deltas_dm)
# _, cls_gas = objectives.vpower_spectrum(deltas_gas)

In [9]:
# gravity_model = NeuralSplineFourierFilterNNX(n_knots=8, d_latent=16, rngs=nnx.Rngs(0))
# gravity_loss_fn = ParticleLoss(mesh_per_dim, w_pos=1.0, w_cls=1.0, loss_type="huber", robust_scale=mesh_per_dim//8, k_max=2)

# checkpoint_file = os.path.join(os.getcwd(), f"checkpoints/gravity_model_code={CODE},parts={parts_per_dim},mesh={mesh_per_dim}.jx")
# # gravity_model.load(checkpoint_file)

## training

In [10]:
# total_steps = 100
# learning_rate = 1e-3
# clip_norm = 1

# gravity_optimizer = nnx.ModelAndOptimizer(
#     gravity_model,
#     optax.chain(
#         optax.clip_by_global_norm(clip_norm),
#         optax.adam(learning_rate)
#     )
# )

# losses = []
# grad_norms = []

In [12]:
# # evaluating the loss for all snapshots is computationally feasible
# for i in (pbar := tqdm.tqdm(range(total_steps))):
#     y0 = (dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0])
#     # t0 = all_scales[0]
#     t0 = scales[0]
#     t1 = scales[-1]
    
#     loss, grad= train_step(
#         gravity_loss_fn,
#         gravity_optimizer, 
#         y0, 
#         t0,
#         ref_t=scales,
#         ref_poss=dm_poss, 
#         ref_cls=cls_dm,
#         ref_deltas=deltas_dm,
#         gravity_model=gravity_model,
#         pressure_model=None,
#         model_to_train="gravity",
#         tstep=scales[(t0 <= scales) & (scales <= t1)],
#         # tstep=all_scales[(t0 <= all_scales) & (all_scales <= t1)],
#         nt=1,
#     )

#     losses.append(loss)
#     pbar.set_description(f"Loss: {loss:.4f}")

# fig, ax = plt.subplots()
# ax.plot(losses)
# ax.set(yscale="log")

  0%|          | 0/100 [00:00<?, ?it/s]

Solving ODE in 67 steps (StepTo with 1 steps between ts)
dark matter and gas
Using learned correction to the gravitational potential
dark matter and gas
Using learned correction to the gravitational potential
dark matter and gas
Using learned correction to the gravitational potential
Using particle loss
w_pos = 1.0, loss = huber, scale = 16.0
w_cls = 1.0, k_max = 2, k_type = step


2025-10-22 04:45:26.277100: E external/xla/xla/service/slow_operation_alarm.cc:73] Constant folding an instruction is taking > 1s:

  %scatter.3093 = f32[65]{0} scatter(%broadcast.156, %constant.2766, %constant.2768), update_window_dims={1}, inserted_window_dims={}, scatter_dims_to_operand_dims={0}, index_vector_dim=1, to_apply=%region_46.3092, metadata={op_name="jit(train_step)/jvp(vmap())/scatter-add" source_file="/global/u2/a/athomsen/flatiron/JaxPM/jaxpm/utils.py" source_line=72}

This isn't necessarily a bug; constant-folding is inherently a trade-off between compilation time and speed at runtime. XLA has some guards that attempt to keep constant folding from taking too long, but fundamentally you'll always be able to come up with an input program that takes a long time.

If you'd like to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
2025-10-22 04:45:31.062402: E external/xla/xla/service/slow_operation_alarm.cc:140] The operation took 5.78540

XlaRuntimeError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 39661385008 bytes.

In [ ]:
# gravity_model.save(checkpoint_file)

In [ ]:
# i_plot = np.linspace(0, scales.shape[0]-1, 4, dtype=int)
# for camels_dict in [train_dict]:
# # for camels_dict in [train_dict, vali_dict]:
#     diagnostics.run_simulations(
#         camels_dict,
#         mesh_per_dim,
#         gravity_model=gravity_model,
#         nt=1,
#         # nt=2,
#         i_init=0,
#         i_plot=i_plot,
#         plot_dm=True,
#         plot_gas=False,
#     )

# pressure correction

## model

### MLP

In [14]:
# pressure_model = MLP(
#     d_in=5 + with_latent,
#     d_out=1 + with_latent, 
#     # d_hidden=16, 
#     d_hidden=64, 
#     # d_hidden=128,
#     # d_hidden=256,
#     n_hidden=4, 
#     # n_hidden=2, 
#     # dropout_rate=0.01,
#     dropout_rate=0.0,
#     rngs=nnx.Rngs(0),
#     norm_type="layer",
#     # norm_type="batch",
#     activation=jax.nn.swish,
#     k_filter="spline",
# )

# # learning_rate = 1e-3
# learning_rate = 1e-4
# # learning_rate = 1e-5

### CNN

In [15]:
pressure_model = ConditionedCNN(
    d_in=4 + with_latent,
    d_out=1 + with_latent,
    # d_hidden=16,
    d_hidden=64,
    n_hidden=2,
    # n_hidden=4,
    kernel_size=(3, 3, 3),
    # kernel_size=(5, 5, 5),
    rngs=nnx.Rngs(0),
    norm_type="layer",
    activation=jax.nn.swish,
    use_residual=True,
)

learning_rate = 1e-4
# learning_rate = 1e-5

In [ ]:
pressure_model = ConditionedCNN(
    d_in=4 + with_latent,
    d_out=1 + with_latent,
    d_hidden=16,
    n_hidden=6,
    kernel_size=(3, 3, 3),
    rngs=nnx.Rngs(0),
    norm_type="layer",
    activation=jax.nn.swish,
    use_residual=True,
)

learning_rate = 1e-4
# learning_rate = 1e-5

In [16]:
# checkpoint_file = os.path.join(os.getcwd(), f"checkpoints/pressure_code={CODE},parts={parts_per_dim},mesh={mesh_per_dim}")
# checkpoint_file += "_cnn_default_v0"
# checkpoint_file += ".jx"
# pressure_model.save(checkpoint_file)

## training

In [17]:
def train_step_wrapper(i0=0, aug_key=None):
    y0 = (dm_poss[i0], dm_vels[i0], gas_poss[i0], gas_vels[i0])
    if with_latent:
        if isinstance(pressure_model, jaxpm.nn.MLP):
            y0 += (jnp.ones((parts_per_dim**3, 1)),)
        elif isinstance(pressure_model, jaxpm.nn.ConditionedCNN):
            y0 += (jnp.ones(mesh_shape + [1]),)
        else:
            raise NotImplementedError
    t0 = scales[i0]

    loss, grad = train_step(
        pressure_loss_fn,
        pressure_optimizer,
        y0,
        t0,
        ref_t=scales[i_ref],
        ref_poss=gas_poss,
        ref_vels=gas_vels,
        ref_cls=cls_gas,
        ref_deltas=deltas_gas,
        pressure_model=pressure_model,
        # gravity_model=gravity_model,
        model_to_train="pressure",
        aug_key=aug_key,
        tstep=scales[(t0 <= scales) & (scales <= scales[i_ref[-1]])],
        nt=1,
        # nt=2,
    )

    losses.append(float(loss))
    grads.append(float(grad))
    pbar.set_description(f"[{i_ref[0]}, {i_ref[-1]}], Loss: {loss:.4e}, Grad: {grad:.4e}")


def plot_training(losses, grad_norms):
    fig, ax = plt.subplots(figsize=(6, 10), nrows=2, sharex=True)
    ax[0].plot(losses)
    ax[0].set(yscale="log", title="loss")

    ax[1].plot(grad_norms)
    ax[1].set(yscale="log", title="norm(grad)")


### loss

In [18]:
pressure_loss_fn = ParticleLoss(
    mesh_per_dim,
    w_pos=1.0,
    w_vel=0.01,
    w_cls=0.1,
    w_cross=1.0,
    w_snapshot=0.0,
    k_max=2,
    loss_type="huber",
    robust_scale=mesh_per_dim//16,
    cutoff_quantile=0.95,
)

In [19]:
# pressure_loss_fn = FieldLoss(
#     mesh_per_dim,
#     w_field=1.0,
#     w_cls=0.1,
#     k_max=2,
#     use_arcsinh=False,
# )

### optimizer

In [20]:
total_steps = 100

# learning_rate = optax.warmup_cosine_decay_schedule(
#     init_value=1e-5,
#     peak_value=1e-4,
#     end_value=0.0,
#     warmup_steps=total_steps//5,
#     decay_steps=total_steps - total_steps//5, 
# )
learning_rate = 1e-5
clip_norm = 1
pressure_optimizer = nnx.ModelAndOptimizer(
    pressure_model,
    optax.chain(
        optax.clip_by_global_norm(clip_norm),
        optax.adam(learning_rate)
    )
)

losses = []
grads = []

## loss snapshots

In [21]:
i0 = 0

### subset

In [22]:
key = jax.random.key(71)

for i in (pbar := tqdm.tqdm(range(total_steps))):
    key, subkey = jax.random.split(key)
    train_step_wrapper(i0=i0, aug_key=subkey)
    
plot_training(losses, grads)

  0%|          | 0/100 [00:00<?, ?it/s]

Applying augmentations (random flips and 90 degree rotations)
Solving ODE in 67 steps (StepTo with 1 steps between ts)
dark matter and gas
Using learned pressure force
No latent variable
dark matter and gas
Using learned pressure force
No latent variable
dark matter and gas
Using learned pressure force
No latent variable
Using particle loss
w_pos = 1.0, loss = huber, scale = 8.0
w_vel = 0.01, loss = huber, scale = 32.0
w_cls = 0.1, k_max = 2, k_type = step
w_cross = 1.0


2025-10-17 05:30:05.411425: E external/xla/xla/service/slow_operation_alarm.cc:73] Constant folding an instruction is taking > 1s:

  %scatter.4991 = f32[65]{0} scatter(%broadcast.209, %constant.9028, %constant.9030), update_window_dims={1}, inserted_window_dims={}, scatter_dims_to_operand_dims={0}, index_vector_dim=1, to_apply=%region_66.4990, metadata={op_name="jit(train_step)/jvp(vmap())/scatter-add" source_file="/global/u2/a/athomsen/flatiron/JaxPM/jaxpm/utils.py" source_line=72}

This isn't necessarily a bug; constant-folding is inherently a trade-off between compilation time and speed at runtime. XLA has some guards that attempt to keep constant folding from taking too long, but fundamentally you'll always be able to come up with an input program that takes a long time.

If you'd like to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
2025-10-17 05:30:09.917301: E external/xla/xla/service/slow_operation_alarm.cc:140] The operation took 5.50612

XlaRuntimeError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 28746733024 bytes.

### checkpointing

In [ ]:
checkpoint_file = os.path.join(os.getcwd(), f"checkpoints/pressure_code={CODE},parts={parts_per_dim},mesh={mesh_per_dim}")
checkpoint_file += "_cnn_default_v0"
checkpoint_file += ".jx"
pressure_model.save(checkpoint_file)

# testing

In [ ]:
i_plot = np.linspace(i0, 33, 4, dtype=int)
# i_plot = np.linspace(i0, 33, 8, dtype=int)
# i_plot = np.linspace(i0, 33, 2, dtype=int)
# i_plot = [-1]
# for camels_dict in [train_dict, vali_dict]:
for camels_dict in [vali_dict]:
    diagnostics.run_simulations(
        camels_dict,
        mesh_per_dim,
        # gravity_model=gravity_model,
        pressure_model=pressure_model, 
        i_init=i0,
        i_plot=i_plot,
        nt=2,
        plot_dm=True,
        plot_gas=True,
        with_latent=with_latent,
        plot_latent=with_latent,
        loss_fn=pressure_loss_fn,
    )

In [ ]:
stop

# temp